In [23]:
import pandas as pd

In [24]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

# Retrieve the key
api_key = os.getenv("OPENAI_API_KEY")

print("API Key loaded:", api_key is not None)

API Key loaded: True


In [26]:
import packaging
from openai import OpenAI
client = OpenAI(api_key = api_key)

### Import and filter dataset based on a minimum number of samples per class

The dataset contains 1588 categories

In [27]:
df_input = pd.read_csv('train.csv')

In [28]:
len(df_input)

20000000

In [29]:
df_input['category'].value_counts()

category
PANTS                                   35973
COFFEE_MAKERS                           35104
BABY_CAR_SEATS                          34163
MUSICAL_KEYBOARDS                       33222
MATTRESSES                              32967
                                        ...  
CONSTRUCTION_LIME_BAGS                    206
COLD_FOOD_AND_DRINK_VENDING_MACHINES      162
PAINTBALL_SMOKE_GRENADES                  154
COMMERCIAL_POPCORN_MACHINES               141
HAMBURGER_FORMERS                         109
Name: count, Length: 1588, dtype: int64

In [30]:
n_sample = 1000
min_samples = 3

# Sample reproducibly
df_sample = df_input.sample(n=n_sample, random_state=42)

# Drop columns
df_sample = df_sample.drop(["language", "label_quality"], axis=1)

# Keep only categories with at least `min_samples`
valid_categories = df_sample['category'].value_counts()
valid_categories = valid_categories[valid_categories >= min_samples].index
df_sample = df_sample[df_sample['category'].isin(valid_categories)]

# Renamdf_input_samplee column
df_sample = df_sample.rename(columns={'title': 'text'})

In [31]:
valid_categories

Index(['KEYBOARD_CONTROLLERS', 'AIR_COMPRESSORS', 'CV_JOINTS',
       'CAMERA_AND_CELLPHONE_STABILIZERS', 'PANTS',
       'AUTOMOTIVE_SPRING_SUSPENSIONS', 'SHAVING_MACHINES',
       'CD_AND_DVD_PLAYERS', 'VEHICLE_STICKERS', 'PERFUMES', 'SANDPAPERS',
       'ROLLER_SKATES', 'FABRICS', 'CLEANING_CLOTHS', 'NETWORK_SWITCHES',
       'DECORATIVE_PAINTINGS', 'BOARD_GAMES', 'MEN_SWIMWEAR',
       'CELLPHONE_AND_TABLET_CASES', 'GLASSES_FRAMES', 'WINES', 'ICE_BUCKETS',
       'DECORATIVE_VINYLS', 'HOME_APPLIANCE_CONTACTORS_AND_RELAYS',
       'HAIR_STRAIGHTENERS', 'DOLLS', 'VEHICLE_CLUTCH_CABLES', 'POUFS',
       'RACQUETS', 'MOTORCYCLE_SEATS', 'AUTOMOTIVE_BATTERIES', 'SEAT_BELTS',
       'EYESHADOWS', 'WRISTWATCHES', 'FISHING_RODS', 'AUTOMOTIVE_WATER_PUMPS',
       'AUTOMOTIVE_CV_JOINT_BOOTS', 'DOORS', 'PROTECTION_NETS',
       'FISHING_REELS', 'VIBRATORS', 'MOTORCYCLE_BATTERIES', 'VESTS',
       'DEEP_FRYERS', 'MUGS', 'COMPLETE_SKATEBOARDS', 'CLASSIC_CARS',
       'ENGINE_OIL_PRESSURE_SENSORS

##### Create some examples to use in the prompt

In [32]:
pd.reset_option('display.max_columns') 
pd.reset_option('display.max_rows')

In [33]:
n_examples = 3
top_categories = df_input['category'].value_counts().head(n_examples).reset_index()
examples = df_input[['title', 'category']].drop_duplicates('category', keep='first')
examples_filtered = examples.merge(top_categories[['category']], on= 'category', how='inner')

In [34]:
top_categories

,category,count
0,PANTS,35973
1,COFFEE_MAKERS,35104
2,BABY_CAR_SEATS,34163


In [35]:
few_examples = ""
for row_number, row in examples_filtered.iterrows():
    few_examples += f"Example {row_number+1} '{row['title']}': {row['category']}\n"

In [36]:
print(few_examples)

Example 1 'Butaca  6 Cuotas Sin Interes!! Para Auto Bebesit Hasta 25kg': BABY_CAR_SEATS
Example 2 'Cafetera De Filtro Hamilton Beach 12 Tazas': COFFEE_MAKERS
Example 3 ' Calça Feminina Pra Mulher Gravidas Dois Bolso Cintura Alta ': PANTS



##### Create a function with the prompt to automate the classification request to AI

- System message: high-level, global instructions about the assistant’s role, style, and constraints.

- User message: the actual classification request, including the text to classify and categories.

In [37]:
def create_classification_prompt(text_to_classify, categories):
    category_list_str = ",".join(categories)
    prompt = (
        f"Here are some examples of classifications:\n"
        f"{few_examples}\n"
        f"Now classify the following MercadoLibre product title into one of these categories: {category_list_str}.\n"
        f"Text: \"{text_to_classify}\"\n"
        f"Classification:"
    )
    return prompt

In [38]:
user_inputs = [create_classification_prompt(title, valid_categories) for title in df_sample['text']]

In [39]:
len(valid_categories), len(df_sample), len(user_inputs[1])

(73, 247, 1583)

In [51]:
user_inputs[:3]

['Here are some examples of classifications:\nExample 1 \'Butaca  6 Cuotas Sin Interes!! Para Auto Bebesit Hasta 25kg\': BABY_CAR_SEATS\nExample 2 \'Cafetera De Filtro Hamilton Beach 12 Tazas\': COFFEE_MAKERS\nExample 3 \' Calça Feminina Pra Mulher Gravidas Dois Bolso Cintura Alta \': PANTS\n\nNow classify the following MercadoLibre product title into one of these categories: KEYBOARD_CONTROLLERS,AIR_COMPRESSORS,CV_JOINTS,CAMERA_AND_CELLPHONE_STABILIZERS,PANTS,AUTOMOTIVE_SPRING_SUSPENSIONS,SHAVING_MACHINES,CD_AND_DVD_PLAYERS,VEHICLE_STICKERS,PERFUMES,SANDPAPERS,ROLLER_SKATES,FABRICS,CLEANING_CLOTHS,NETWORK_SWITCHES,DECORATIVE_PAINTINGS,BOARD_GAMES,MEN_SWIMWEAR,CELLPHONE_AND_TABLET_CASES,GLASSES_FRAMES,WINES,ICE_BUCKETS,DECORATIVE_VINYLS,HOME_APPLIANCE_CONTACTORS_AND_RELAYS,HAIR_STRAIGHTENERS,DOLLS,VEHICLE_CLUTCH_CABLES,POUFS,RACQUETS,MOTORCYCLE_SEATS,AUTOMOTIVE_BATTERIES,SEAT_BELTS,EYESHADOWS,WRISTWATCHES,FISHING_RODS,AUTOMOTIVE_WATER_PUMPS,AUTOMOTIVE_CV_JOINT_BOOTS,DOORS,PROTECTION_NE

#### Create functions to build the message and to get the responses

Create a function to build the message in the expected format by the OpenAI Chat API and another to call the model and extract the responses

The message has two roles: system and user

In [44]:
def format_message(role, content):
    return {"role": role, "content": content}


def get_response(messages):
    completion = client.chat.completions.create(
        model="gpt-5-mini",
        messages=messages
    )
    content = completion.choices[0].message.content
    return content

In [45]:
system_message = (
"You are a product classifier for MercadoLibre product titles. "
"Titles may be in Spanish or Portuguese. "
"Always classify them into one of the provided categories, which are in English. "
"Return only the category name, without explanations or extra text. "
"If no category seems perfect, choose the closest match from the list. "
"Do not invent new categories."
)

##### Set messages using the system_message and the user_inputs as output of create_classification_prompt() function.

In [46]:
responses = []
for user_input in user_inputs:
    messages = [format_message("system", system_message),
                format_message("user", user_input)]
    response = get_response(messages)
    responses.append(response)

In [47]:
responses[:5]

['CLASSIC_CARS',
 'MOTORCYCLE_IGNITION_SWITCHES',
 'NETWORK_SWITCHES',
 'CELLPHONES',
 'AUTOMOTIVE_BATTERIES']

In [48]:
df_sample['prediction'] = responses

In [49]:
df_sample.head()

,text,category,prediction
19555084,Nissan Versa,CLASSIC_CARS,CLASSIC_CARS
831796,Llave Contacto Yamaha Fz 16 + Tapa Tanque Mpr,MOTORCYCLE_IGNITION_SWITCHES,MOTORCYCLE_IGNITION_SWITCHES
6623530,Switch Ubiquiti Unifi 8 - Puertos Giga Admini...,NETWORK_SWITCHES,NETWORK_SWITCHES
8310326,Moto G5 Libre Huella Zona Oeste,CELLPHONES,CELLPHONES
14784082,Bateria Heliar 50 Jd *** New Civic- Crv***bate...,AUTOMOTIVE_BATTERIES,AUTOMOTIVE_BATTERIES


In [50]:
from sklearn.metrics import accuracy_score, f1_score

print("Accuracy: ", accuracy_score(df_sample['category'], df_sample['prediction']))
print("F1 Score: ", f1_score(df_sample['category'], df_sample['prediction'], average='macro')) #macro: (treat all classes equally)

Accuracy:  0.9190283400809717
F1 Score:  0.9150339006503391


#### Check tokens to estimate costs

247 records with 73 categories → 569 tokens per record

569×247=140,543 tokens

| Model       | Price per 1k tokens | Cost                                    |
| ----------- | ------------------- | --------------------------------------- |
| GPT-5-nano  | $0.0001             | 140,543 ÷ 1,000 × 0.0001 ≈ **$0.014**   |
| GPT-4o-mini | $0.15 / 1M          | 140,543 ÷ 1,000,000 × 0.15 ≈ **$0.021** |
| GPT-4o      | $2.50 / 1M          | 140,543 ÷ 1,000,000 × 2.50 ≈ **$0.35**  |
| GPT-4-turbo | $10 / 1M            | 140,543 ÷ 1,000,000 × 10 ≈ **$1.41**    |


The whole dataset:

1588 categories: 1,588 × 6.22 ≈ 9,874 tokens for categories

Text/instructions part: 9,874 + 115 ≈ 9,989 ≈10,000 tokens per record


Total tokens for 20,000,000 records ≈ 10,000×20,000,000 = 200,000,000,000 tokens

**Cost for 20M records** ≈ 200,000,000,000 ÷ 1,000 × 0.0001 = **$20,000** 😱 

? Fine Tuning GPT: This process adapts a pre-trained GPT model to a specific task by training it on your labeled data.

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4o-mini")

text = user_inputs[1]
categories_str = ", ".join(valid_categories)
tokens_text = encoding.encode(text)
tokens_categories = encoding.encode(categories_str)

print("Number of Categories:", len(valid_categories))
print("Tokens of Categories", len(tokens_categories))
print("Tokens of Full Text (instructions + examples + categories + product title):", len(tokens_text))

Number of Categories: 73
Tokens of Categories 454
Tokens of Full Text (instructions + examples + categories + product title): 569
